In [ ]:
class EnergyStorageEnv(gym.Env):
    def __init__(self, num_agents, episode_length, future_price, battery_capacity, max_charge_rate,future_load, w_pv_rolling, data_path=None, mode='train'):
        # --- 核心属性 ---
        self.num_agents = num_agents
        self.episode_length = episode_length
        self.future_price = future_price
        self.future_load = future_load
        self.w_pv_rolling = w_pv_rolling #滚动峰谷差奖励的权重
        self.mode = mode
        self.n = self.num_agents
        
        # --- 储能相关参数 ---
        self.battery_capacity = battery_capacity
        self.max_charge_rate = max_charge_rate
        self.efficiency = 1  # 充放电效率为0.95
        self.init_soc = 0.05
        
        # --- 数据加载与处理 ---
        if data_path is None:
            data_path = Path.cwd().parent / "data" / "train_prices.csv"
        self.raw_data = self._load_data(data_path)

       # 计算全局负荷标准化参数
        load_cols = [f"load{i+1}" for i in range(self.num_agents)]
        all_loads = self.raw_data[load_cols].values.flatten()
        self.load_norm_params = {'mean': np.mean(all_loads), 'std': np.std(all_loads)}
        self.num_available_episodes = (len(self.raw_data) // self.episode_length) - 1

        # --- 状态与历史记录 ---
        self.current_step = 0
        self.battery_soc = {}
        self.load_profiles = {}
        self.net_load_history = deque(maxlen=96)
        # --- Gym 接口 ---
        self.observation_space = self._create_observation_space()
        self.action_space = self._create_action_space()
        # 初始化状态
        self.reset()

    def _create_observation_space(self) -> List[spaces.Box]:
        obs_shape = (5 + self.future_price + self.future_load,)
        return [spaces.Box(low=-np.inf, high=np.inf, shape=obs_shape, dtype=np.float32) for _ in range(self.n)]

    def _create_action_space(self) -> List[spaces.Box]:
        return [spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32) for _ in range(self.n)]
    
    def _load_data(self, data_path: str) -> pd.DataFrame:
        try:
            return pd.read_csv(data_path)
        except Exception as e:
            raise ValueError(f"Unable to load data file: {e}")
       
    def _get_single_observation(self, agent_id: int) -> np.ndarray:
        time_sin = np.sin(2 * np.pi * self.current_step / self.episode_length)
        time_cos = np.cos(2 * np.pi * self.current_step / self.episode_length)
        
        profile = self.load_profiles[agent_id]
        current_price = profile['price'].iloc[self.current_step]
        current_load = profile['load'].iloc[self.current_step]
        current_soc = self.battery_soc[agent_id]
        
        # 为未来观测准备数据，如果到达末尾则用0填充
        future_prices = profile['price'].iloc[self.current_step + 1 : self.current_step + 1 + self.future_price].values
        future_loads = profile['load'].iloc[self.current_step + 1 : self.current_step + 1 + self.future_load].values
        
        padded_prices = np.pad(future_prices, (0, self.future_price - len(future_prices)), 'constant')
        padded_loads = np.pad(future_loads, (0, self.future_load - len(future_loads)), 'constant')

        obs = np.concatenate([
            [time_sin, time_cos, current_price, current_load, current_soc],
            padded_prices,
            padded_loads
        ]).astype(np.float32)
        return obs

    def reset(self, episode_idx=None) -> List[np.ndarray]:
        """重置环境。如果提供了episode_idx，则用于测试模式。"""
        self.current_step = 0
        self.battery_soc = {i: self.init_soc for i in range(self.n)}
        self.net_load_history.clear()

        if episode_idx is None:
            # 训练模式：随机选择一天（确保有前一天的数据）
            day_idx = np.random.randint(1, self.num_available_episodes + 1)
        else:
            # 测试模式：使用指定索引（确保有前一天的数据）
            day_idx = episode_idx + 1
            if day_idx > self.num_available_episodes:
                raise IndexError(f"Test episode_idx {episode_idx} is out of valid range.")

        # --- 计算电价标准化参数 (基于前一天) ---
        prev_day_start = (day_idx - 1) * self.episode_length
        prev_day_end = prev_day_start + 2*self.episode_length
        prev_day_prices = self.raw_data['price'].iloc[prev_day_start:prev_day_end].values
        self.episode_price_mean = np.mean(prev_day_prices)
        self.episode_price_std = np.std(prev_day_prices)
        if self.episode_price_std == 0: self.episode_price_std = 1

        # --- 准备当前回合的数据 ---
        start = day_idx * self.episode_length
        end = start + self.episode_length
        
        episode_prices = self.raw_data['price'].iloc[start:end].values
        normalized_prices = (episode_prices - self.episode_price_mean) / self.episode_price_std
        
        load_cols = [f"load{i+1}" for i in range(self.num_agents)]
        for agent_id in range(self.n):
            agent_load = self.raw_data[load_cols[agent_id]].iloc[start:end].values
            normalized_load = (agent_load - self.load_norm_params['mean']) / self.load_norm_params['std']
            self.load_profiles[agent_id] = pd.DataFrame({'load': normalized_load, 'price': normalized_prices})
        
        # 初始化前一天的总负荷历史用于削峰填谷奖励计算
        prev_day_total_load = self.raw_data[load_cols].iloc[prev_day_start:prev_day_end].sum(axis=1).values
        self.net_load_history.extend(prev_day_total_load)

        return [self._get_single_observation(i) for i in range(self.n)]
    
    def step(self, actions: List[np.ndarray]) -> Tuple[List[np.ndarray], List[float], List[bool], Dict]:
        """执行一个时间步的环境动态"""
        # [MODIFIED] 兼容Runner: 将输入的动作列表转换为字典
        actions_dict = {i: action[0] for i, action in enumerate(actions)}
        current_price_normalized = self.load_profiles[0]['price'].iloc[self.current_step]
        real_price = current_price_normalized * self.episode_price_std + self.episode_price_mean

        rewards, ess_power = {}, {}
        total_ess_power, total_economic_reward, total_soc_penalty = 0, 0, 0

        for agent_id, action_val in actions_dict.items():
            # 动作值在[-1, 1]之间，映射到[-max_rate, max_rate]
            charge_power = action_val * self.max_charge_rate
            delta_soc = (charge_power * self.efficiency if charge_power >= 0 else charge_power / self.efficiency) / self.battery_capacity
            new_soc = self.battery_soc[agent_id] + delta_soc

            # 初始化越限惩罚项
            soc_penalty = 0.0
            if new_soc > 0.95:
                soc_penalty = -5.0 if action_val > 0 else 0
                charge_power = min(charge_power, ((0.95 - self.battery_soc[agent_id]) * self.battery_capacity) / self.efficiency)
                new_soc = 0.95
            elif new_soc < 0.05:
                soc_penalty = -5.0 if action_val < 0 else 0
                charge_power = max(charge_power, -((self.battery_soc[agent_id] - 0.05) * self.battery_capacity * self.efficiency))
                new_soc = 0.05

            self.battery_soc[agent_id] = new_soc
            ess_power[agent_id] = charge_power
            total_ess_power += charge_power
            total_soc_penalty += soc_penalty

            # 计算奖励
            # 当充电时，经济收益为负；当放电时，经济收益为正
            economic_reward = -charge_power * real_price
            total_economic_reward += economic_reward
            rewards[agent_id] = economic_reward + soc_penalty

        # --- 计算协同奖励 ---
        # 1. 计算当前时间步的真实原始总负荷
        original_total_load = 0
        for agent_id in range(self.num_agents):
            normalized_load = self.load_profiles[agent_id]['load'].iloc[self.current_step]
            real_load = normalized_load * self.load_norm_params['std'] + self.load_norm_params['mean']
            original_total_load += real_load
        # 2. 计算净总负荷
        net_total_load = original_total_load + total_ess_power
        self.net_load_history.append(net_total_load)
        
        # 3. 计算滚动峰谷差协同奖励
        peak_valley_penalty = 0
        if self.w_pv_rolling > 0 and len(self.net_load_history) > 1:
            peak_valley_penalty = -self.w_pv_rolling * np.std(self.net_load_history)
            for agent_id in range(self.n):
                rewards[agent_id] += peak_valley_penalty / self.n


        self.current_step += 1
        
        # 检查episode是否结束
        done = self.current_step >= self.episode_length
        
        obs_n = [self._get_single_observation(i) for i in range(self.n)] if not done else [np.zeros_like(self.observation_space[i].low) for i in range(self.n)]
        reward_n = [rewards[i] for i in range(self.n)]
        done_n = [done] * self.n
      
        # 将需要记录的内部状态放入info字典
        info = {
            'ess_power': ess_power, 'soc': self.battery_soc, 'price': real_price,
            'original_total_load': original_total_load, 'net_total_load': net_total_load,
            'original_load_cost': original_total_load * real_price,
            'cost_with_ess': net_total_load * real_price,
            'ess_profit': total_economic_reward, 'soc_penalty': total_soc_penalty,
            'peak_valley_penalty': peak_valley_penalty
        }
        return obs_n, reward_n, done_n, info

    def close(self):
        pass